In [1]:
dpii = 96

In [ ]:
# 하나의 클래스 이미지 탐지 폴더에 있는 이미지 전부 변환 + 전처리 형태로 변환

# fire

import torch
from pathlib import Path

#################################### For Image ####################################
from PIL import Image
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# 경로와 프롬프트 설정
input_dir = Path(
    r"C:\Users\SSAFY\Desktop\selected_data\train\origin\DetectiumFire\images"
)
output_dir = Path(
    r"C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\fire"
)
# YOLO 라벨 저장 폴더
label_dir = Path(
    r"C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\fire\label"
)

label_dir.mkdir(parents=True, exist_ok=True)

# 클래스 번호
# person=0, fire=1, smoke=2
class_id = 1
prompt = "fire"

confidence_threshold = 0.4
mask_threshold = 1

# 결과 폴더 생성
output_dir.mkdir(parents=True, exist_ok=True)

# Load the model
model = build_sam3_image_model("./sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz")
processor = Sam3Processor(
    model,
    confidence_threshold=confidence_threshold,
)
# GPU가 BF16을 지원하면 BF16, 아니면 FP16
amp_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

# 입력 폴더의 이미지 목록
image_paths = sorted(
    path
    for path in input_dir.iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
)

for image_path in image_paths:
    # Load an image
    image = Image.open(image_path).convert("RGB")

    # SAM3 추론
    with torch.inference_mode():
        with torch.autocast(
            device_type="cuda",
            dtype=amp_dtype,
        ):
            inference_state = processor.set_image(image)

            output = processor.set_text_prompt(
                state=inference_state,
                prompt=prompt,
            )

            output["masks"] = output["masks_logits"] > mask_threshold

    # Get the masks, bounding boxes, and scores
    masks, boxes, scores = output["masks"], output["boxes"], output["scores"]

    # -----------------------------------------------------------------------------

    import matplotlib.pyplot as plt

    # 원본 이미지 표시
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(image)

    # 박스만 표시
    for box in boxes.detach().cpu().tolist():
        x1, y1, x2, y2 = box

        rectangle = plt.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            edgecolor="blue",
            linewidth=2,
        )

        ax.add_patch(rectangle)

    ax.axis("off")
    fig.tight_layout(pad=0)

    # 결과 저장
    save_path = output_dir / f"{image_path.stem}_sam3_result.png"

    fig.savefig(
        save_path,
        dpi=dpii,
        bbox_inches="tight",
        pad_inches=0,
    )

    plt.close(fig)
    print(f"저장 완료: {save_path}")

저장 완료: C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\fire\240_F_182451272_P5k36aySV2mrcHqRvREOgkrKDttvpbw5_jpg.rf.c5461783b371823215bbd0bf856d751b_sam3_result.png
저장 완료: C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\fire\240_F_183228086_MWMXWnF7WI4zRRRqrk2WMpRAuZjL97TM_jpg.rf.d9eb4fa09473ada87b1acbee5208f406_sam3_result.png
저장 완료: C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\fire\240_F_191639767_FD6azNs7Bgj3n7q4rDFf3qpNKOs536o9_jpg.rf.022b4512546b3f03cee99d33015a4992_sam3_result.png
저장 완료: C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\fire\240_F_191640296_0CrTrQQysTIsMtbY6sUY4N8WbL1fjFFZ_jpg.rf.e9981f3ceb1644bdc35dd14e8c1f70e5_sam3_result.png
저장 완료: C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\fire\240_F_191640350_6rrGlV7UlOFXZkPyoxdDxvYa1YN3nNtk_jpg.rf.7861d1848d9db379af4522206530d594_sam3_result.png
저장 완료: C:\Users\SSAFY\Desktop\selected_data\train\sam3_detec

In [4]:
# 하나의 클래스 이미지 탐지
# 폴더에 있는 이미지 전부 변환 + YOLO 라벨 생성

# fire

import torch
from pathlib import Path

#################################### For Image ####################################
from PIL import Image
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# 경로와 프롬프트 설정
input_dir = Path(
    r"C:\Users\SSAFY\Desktop\selected_data\train\origin\DetectiumFire\images"
)

# SAM3 탐지 결과 확인용 이미지 저장 폴더
output_dir = Path(
    r"C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\fire\image"
)

# YOLO 라벨 저장 폴더
# images 폴더와 같은 위치의 labels 폴더
label_dir = output_dir.parent / "label"

label_dir.mkdir(parents=True, exist_ok=True)

# 클래스 번호
# person=0, fire=1, smoke=2
class_id = 1
prompt = "fire"

confidence_threshold = 0.4
mask_threshold = 1

# 결과 폴더 생성
output_dir.mkdir(parents=True, exist_ok=True)

# Load the model
model = build_sam3_image_model(
    "./sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
)

processor = Sam3Processor(
    model,
    confidence_threshold=confidence_threshold,
)

# GPU가 BF16을 지원하면 BF16, 아니면 FP16
amp_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

# 입력 폴더의 이미지 목록
image_paths = sorted(
    path
    for path in input_dir.iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
)

for image_path in image_paths:
    # Load an image
    image = Image.open(image_path).convert("RGB")

    # SAM3 추론
    with torch.inference_mode():
        with torch.autocast(
            device_type="cuda",
            dtype=amp_dtype,
        ):
            inference_state = processor.set_image(image)

            output = processor.set_text_prompt(
                state=inference_state,
                prompt=prompt,
            )

            output["masks"] = output["masks_logits"] > mask_threshold

    # Get the masks, bounding boxes, and scores
    masks, boxes, scores = (
        output["masks"],
        output["boxes"],
        output["scores"],
    )

    # -------------------------------------------------------------------------
    # YOLO 라벨 생성
    # 형식: class_id x_center y_center width height
    # -------------------------------------------------------------------------

    image_width, image_height = image.size

    label_path = label_dir / f"{image_path.stem}.txt"

    box_list = boxes.detach().cpu().tolist()
    score_list = scores.detach().cpu().tolist()

    # 라벨에 실제로 저장된 객체
    saved_objects = []

    with label_path.open("w", encoding="utf-8") as label_file:
        for box, score in zip(box_list, score_list):
            x1, y1, x2, y2 = box

            # 원래 박스 좌표는 결과 이미지 표시용으로 보관
            original_box = box

            # 이미지 영역을 벗어난 좌표 보정
            x1 = max(0.0, min(x1, image_width))
            y1 = max(0.0, min(y1, image_height))
            x2 = max(0.0, min(x2, image_width))
            y2 = max(0.0, min(y2, image_height))

            box_width = x2 - x1
            box_height = y2 - y1

            if box_width <= 0 or box_height <= 0:
                continue

            # YOLO 형식으로 정규화
            x_center = ((x1 + x2) / 2) / image_width
            y_center = ((y1 + y2) / 2) / image_height
            normalized_width = box_width / image_width
            normalized_height = box_height / image_height

            label_file.write(
                f"{class_id} "
                f"{x_center:.6f} "
                f"{y_center:.6f} "
                f"{normalized_width:.6f} "
                f"{normalized_height:.6f}\n"
            )

            # 라벨에 저장된 순서대로 객체와 확률 보관
            saved_objects.append(
                (original_box, float(score))
            )

    print(f"라벨 저장 완료: {label_path}")

    # -------------------------------------------------------------------------

    import matplotlib.pyplot as plt

    # 원본 이미지 표시
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(image)

    # 박스 + 객체 순서 + 확률 표시
    for object_number, (box, score) in enumerate(
        saved_objects,
        start=1,
    ):
        x1, y1, x2, y2 = box

        rectangle = plt.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            edgecolor="blue",
            linewidth=2,
        )

        ax.add_patch(rectangle)

        # 예: fire1 0.87
        object_text = f"{prompt}{object_number} {score:.2f}"

        ax.text(
            x1,
            max(y1 - 5, 0),
            object_text,
            color="white",
            fontsize=10,
            bbox={
                "facecolor": "blue",
                "alpha": 0.8,
                "edgecolor": "none",
                "pad": 2,
            },
        )
        ax.axis("off")
        fig.tight_layout(pad=0)

    # 결과 저장
    save_path = output_dir / f"{image_path.stem}.png"

    fig.savefig(
        save_path,
        dpi=dpii,
        bbox_inches="tight",
        pad_inches=0,
    )

    plt.close(fig)
    print(f"이미지 저장 완료: {save_path}")

라벨 저장 완료: C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\fire\label\240_F_182451272_P5k36aySV2mrcHqRvREOgkrKDttvpbw5_jpg.rf.c5461783b371823215bbd0bf856d751b.txt
이미지 저장 완료: C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\fire\image\240_F_182451272_P5k36aySV2mrcHqRvREOgkrKDttvpbw5_jpg.rf.c5461783b371823215bbd0bf856d751b.png
라벨 저장 완료: C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\fire\label\240_F_183228086_MWMXWnF7WI4zRRRqrk2WMpRAuZjL97TM_jpg.rf.d9eb4fa09473ada87b1acbee5208f406.txt
이미지 저장 완료: C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\fire\image\240_F_183228086_MWMXWnF7WI4zRRRqrk2WMpRAuZjL97TM_jpg.rf.d9eb4fa09473ada87b1acbee5208f406.png
라벨 저장 완료: C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\fire\label\240_F_191639767_FD6azNs7Bgj3n7q4rDFf3qpNKOs536o9_jpg.rf.022b4512546b3f03cee99d33015a4992.txt
이미지 저장 완료: C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\Detec

In [4]:
# smoke

import torch
from pathlib import Path

#################################### For Image ####################################
from PIL import Image
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# 경로와 프롬프트 설정
input_dir = Path(
    r"C:\Users\SSAFY\Desktop\selected_data\train\origin\DetectiumFire\images"
)
output_dir = Path(
    r"C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\smoke"
)
prompt = "smoke"

confidence_threshold = 0.35
mask_threshold = 1

# 결과 폴더 생성
output_dir.mkdir(parents=True, exist_ok=True)

# Load the model
model = build_sam3_image_model("./sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz")
processor = Sam3Processor(
    model,
    confidence_threshold=confidence_threshold,
)
# GPU가 BF16을 지원하면 BF16, 아니면 FP16
amp_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

# 입력 폴더의 이미지 목록
image_paths = sorted(
    path
    for path in input_dir.iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
)

for image_path in image_paths:
    # Load an image
    image = Image.open(image_path).convert("RGB")

    # SAM3 추론
    with torch.inference_mode():
        with torch.autocast(
            device_type="cuda",
            dtype=amp_dtype,
        ):
            inference_state = processor.set_image(image)

            output = processor.set_text_prompt(
                state=inference_state,
                prompt=prompt,
            )

            output["masks"] = output["masks_logits"] > mask_threshold

    # Get the masks, bounding boxes, and scores
    masks, boxes, scores = output["masks"], output["boxes"], output["scores"]

    # -----------------------------------------------------------------------------

    import matplotlib.pyplot as plt

    # 원본 이미지 표시
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(image)

    # 박스만 표시
    for box in boxes.detach().cpu().tolist():
        x1, y1, x2, y2 = box

        rectangle = plt.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            edgecolor="blue",
            linewidth=2,
        )

        ax.add_patch(rectangle)

    ax.axis("off")
    fig.tight_layout(pad=0)

    # 결과 저장
    save_path = output_dir / f"{image_path.stem}_sam3_result.png"

    fig.savefig(
        save_path,
        dpi=dpii,
        bbox_inches="tight",
        pad_inches=0,
    )

    plt.close(fig)
    print(f"저장 완료: {save_path}")

저장 완료: C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\smoke\240_F_182451272_P5k36aySV2mrcHqRvREOgkrKDttvpbw5_jpg.rf.c5461783b371823215bbd0bf856d751b_sam3_result.png
저장 완료: C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\smoke\240_F_183228086_MWMXWnF7WI4zRRRqrk2WMpRAuZjL97TM_jpg.rf.d9eb4fa09473ada87b1acbee5208f406_sam3_result.png
저장 완료: C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\smoke\240_F_191639767_FD6azNs7Bgj3n7q4rDFf3qpNKOs536o9_jpg.rf.022b4512546b3f03cee99d33015a4992_sam3_result.png
저장 완료: C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\smoke\240_F_191640296_0CrTrQQysTIsMtbY6sUY4N8WbL1fjFFZ_jpg.rf.e9981f3ceb1644bdc35dd14e8c1f70e5_sam3_result.png
저장 완료: C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\smoke\240_F_191640350_6rrGlV7UlOFXZkPyoxdDxvYa1YN3nNtk_jpg.rf.7861d1848d9db379af4522206530d594_sam3_result.png
저장 완료: C:\Users\SSAFY\Desktop\selected_data\train\sam3_

In [ ]:
# person

import torch
from pathlib import Path

#################################### For Image ####################################
from PIL import Image
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# 경로와 프롬프트 설정
input_dir = Path(
    r"C:\Users\SSAFY\Desktop\selected_data\train\origin\DetectiumFire\images"
)
output_dir = Path(
    r"C:\Users\SSAFY\Desktop\selected_data\train\sam3_detected\DetectiumFire\person"
)
prompt = "person"

confidence_threshold = 0.5
mask_threshold = 1

# 결과 폴더 생성
output_dir.mkdir(parents=True, exist_ok=True)

# Load the model
model = build_sam3_image_model("./sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz")
processor = Sam3Processor(
    model,
    confidence_threshold=confidence_threshold,
)
# GPU가 BF16을 지원하면 BF16, 아니면 FP16
amp_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

# 입력 폴더의 이미지 목록
image_paths = sorted(
    path
    for path in input_dir.iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
)

for image_path in image_paths:
    # Load an image
    image = Image.open(image_path).convert("RGB")

    # SAM3 추론
    with torch.inference_mode():
        with torch.autocast(
            device_type="cuda",
            dtype=amp_dtype,
        ):
            inference_state = processor.set_image(image)

            output = processor.set_text_prompt(
                state=inference_state,
                prompt=prompt,
            )

            output["masks"] = output["masks_logits"] > mask_threshold

    # Get the masks, bounding boxes, and scores
    masks, boxes, scores = output["masks"], output["boxes"], output["scores"]

    # -----------------------------------------------------------------------------

    import matplotlib.pyplot as plt

    # 원본 이미지 표시
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(image)

    # 박스만 표시
    for box in boxes.detach().cpu().tolist():
        x1, y1, x2, y2 = box

        rectangle = plt.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            edgecolor="blue",
            linewidth=2,
        )

        ax.add_patch(rectangle)

    ax.axis("off")
    fig.tight_layout(pad=0)

    # 결과 저장
    save_path = output_dir / f"{image_path.stem}_sam3_result.png"

    fig.savefig(
        save_path,
        dpi=dpii,
        bbox_inches="tight",
        pad_inches=0,
    )

    plt.close(fig)
    print(f"저장 완료: {save_path}")